# Structured interpretation → deterministic policy retrieval

Run cells in order. Edit `user_query` below. The Ollama server must be running and the selected model available. This experiment retrieves policy; it does not execute reservation operations.

In [1]:
import sys

print(sys.executable)

/Users/cahree/dev/Odysseus/.venv/bin/python3


In [2]:
from typing import Annotated, Any, Literal

from pydantic import BaseModel, ConfigDict, Field, model_validator


class SchemaModel(BaseModel):
    model_config = ConfigDict(extra="forbid")


EntityType = Literal["Passenger", "Reservation"]

PropertyName = Literal[
    "id",
    "name",
    "reservation_id",
    "status",
    "refundable",
    "hours_until_departure",
]

ALLOWED_PROPERTIES = {
    "Passenger": {"id", "name"},
    "Reservation": {
        "reservation_id",
        "status",
        "refundable",
        "hours_until_departure",
    },
}


class EntityReference(SchemaModel):
    alias: str = Field(description="Unique query-local name, e.g. reservation")
    type: EntityType
    identifier: str | None = Field(
        default=None,
        description="Known entity ID; null when unresolved. Never invent an ID.",
    )


class PropertyReference(SchemaModel):
    entity: str = Field(description="Alias of an entity in this query")
    property: PropertyName


class Constraint(SchemaModel):
    target: PropertyReference
    operator: Literal["eq", "neq", "gt", "lt", "gte", "lte"]
    value: Any = Field(
        description="Literal comparison value explicitly supplied by the user"
    )


class RetrieveOperation(SchemaModel):
    intent: Literal["retrieve"]
    action: Literal["lookup_reservation", "check_cancellation_eligibility"]
    target: str = Field(description="Alias of the target reservation")
    requested_properties: list[PropertyReference] = Field(default_factory=list)


class ExecuteOperation(SchemaModel):
    intent: Literal["execute"]
    action: Literal["cancel_reservation"]
    target: str = Field(description="Alias of the target reservation")


class Clarification(SchemaModel):
    field: str = Field(
        description="Missing or ambiguous field, e.g. entities.reservation.identifier"
    )
    question: str


class InterpretedQuery(SchemaModel):
    goal: str = Field(description="Short description of the user's desired outcome")

    operation: Annotated[
        RetrieveOperation | ExecuteOperation,
        Field(discriminator="intent"),
    ]

    entities: list[EntityReference] = Field(min_length=1)

    constraints: list[Constraint] = Field(
        default_factory=list,
        description=(
            "User-specified filters, combined with AND. "
            "Do not insert policy requirements or inferred database facts."
        ),
    )

    clarifications: list[Clarification] = Field(default_factory=list)

    @model_validator(mode="after")
    def validate_references(self):
        entities = {entity.alias: entity for entity in self.entities}

        if len(entities) != len(self.entities):
            raise ValueError("Entity aliases must be unique")

        target = entities.get(self.operation.target)
        if target is None or target.type != "Reservation":
            raise ValueError("Operation target must reference a Reservation")

        references = [constraint.target for constraint in self.constraints]
        if isinstance(self.operation, RetrieveOperation):
            references.extend(self.operation.requested_properties)

        for reference in references:
            entity = entities.get(reference.entity)
            if entity is None:
                raise ValueError(f"Unknown entity alias: {reference.entity}")

            if reference.property not in ALLOWED_PROPERTIES[entity.type]:
                raise ValueError(
                    f"{reference.property} is not a property of {entity.type}"
                )

        return self


In [3]:
QUERY_PROMPT = """You are a query interpreter for an Airline Customer Service Agent.
Translate the user's request into the provided InterpretedQuery JSON schema.
Reservation properties: reservation_id, status (confirmed or cancelled),
refundable (boolean), hours_until_departure (number).
Passenger properties: id, name.
Use lookup_reservation for reservation information questions,
check_cancellation_eligibility for questions about whether cancellation is possible,
and cancel_reservation only for requests to perform cancellation.
Do not generate execution plans or tool calls. Do not invent identifiers,
reservation facts, policy constraints, or authorization.
Use null for an unknown identifier and add a clarification question.
Only include constraints explicitly specified by the user.
Return only JSON matching the schema.
"""

MODEL = "qwen3:8b-q4_K_M"
user_query = "Can I cancel reservation ABC123?"


In [4]:
from pathlib import Path

# Locate the project whether Jupyter starts here or at the repository root.
for parent in (Path.cwd(), *Path.cwd().parents):
    if (parent / "experiments/airline/policy_graph.py").is_file():
        if str(parent) not in sys.path:
            sys.path.insert(0, str(parent))
        break
else:
    raise RuntimeError("Start Jupyter inside the Odysseus project")

from experiments.airline.policy_graph import build_policy_graph
from experiments.airline.policy_cli import describe


class PolicyRetrievalError(RuntimeError):
    pass


class RetrievedCondition(SchemaModel):
    node_id: str
    properties: dict[str, Any]
    checked_attributes: list[str]


class PolicyContext(SchemaModel):
    operation: str
    action_node: str
    target_alias: str
    conditions: list[RetrievedCondition]
    source_references: list[str]


ACTION_NODES = {
    "lookup_reservation": "LookupReservation",
    "check_cancellation_eligibility": "CancelReservation",
    "cancel_reservation": "CancelReservation",
}

EXPECTED_CONDITIONS = {
    "LookupReservation": {"Exists"},
    "CancelReservation": {
        "Exists", "IsConfirmed", "IsRefundable",
        "DepartureBeyond24Hours", "HasAuthorization",
    },
}


def retrieve_policies(query: InterpretedQuery, graph) -> PolicyContext:
    operation = query.operation.action
    action_node = ACTION_NODES.get(operation)
    if action_node is None:
        raise PolicyRetrievalError(f"Unsupported operation: {operation}")

    action = describe(graph, action_node)
    if action.get("properties", {}).get("type") != "Action":
        raise PolicyRetrievalError(f"Missing action node: {action_node}")

    required_nodes = {
        edge["target"] for edge in action["outgoing"]
        if edge["relation"] == "REQUIRES"
    }
    expected = EXPECTED_CONDITIONS[action_node]
    if required_nodes != expected:
        raise PolicyRetrievalError(
            f"Policy coverage mismatch for {action_node}: "
            f"missing={sorted(expected - required_nodes)}, "
            f"unexpected={sorted(required_nodes - expected)}"
        )

    conditions = []
    references = {action_node}
    for node_id in sorted(required_nodes):
        condition = describe(graph, node_id)
        properties = condition.get("properties", {})
        if properties.get("type") != "Condition":
            raise PolicyRetrievalError(f"Invalid condition: {node_id}")
        predicate = properties.get("predicate")
        has_predicate = isinstance(predicate, str) and bool(predicate.strip())
        has_comparison = (
            isinstance(properties.get("attribute"), str)
            and properties.get("operator") in {"==", "!=", ">", "<", ">=", "<="}
            and "value" in properties
        )
        if not (has_predicate or has_comparison):
            raise PolicyRetrievalError(f"Condition has no supported rule: {node_id}")

        attributes = sorted({
            edge["target"] for edge in condition["outgoing"]
            if edge["relation"] == "CHECKS"
        })
        if not attributes:
            raise PolicyRetrievalError(f"Missing CHECKS relation: {node_id}")
        for attribute in attributes:
            if graph.nodes[attribute].get("type") != "Attribute":
                raise PolicyRetrievalError(f"Invalid checked attribute: {attribute}")
        if has_comparison and properties["attribute"] not in attributes:
            raise PolicyRetrievalError(f"Comparison disagrees with CHECKS: {node_id}")

        conditions.append(RetrievedCondition(
            node_id=node_id, properties=properties, checked_attributes=attributes,
        ))
        references.add(node_id)
        references.update(attributes)

    return PolicyContext(
        operation=operation, action_node=action_node,
        target_alias=query.operation.target, conditions=conditions,
        source_references=sorted(references),
    )


policy_graph = build_policy_graph()


In [5]:
from ollama import Client

client = Client()


def interpret_and_retrieve(user_query: str, client, graph):
    response = client.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": QUERY_PROMPT},
            {"role": "user", "content": user_query},
        ],
        format=InterpretedQuery.model_json_schema(),
        think=False,
    )
    interpreted = InterpretedQuery.model_validate_json(response.message.content)
    # Mandatory controller-owned retrieval, not an optional model tool call.
    # Either both validated objects are returned, or this function raises.
    context = retrieve_policies(interpreted, graph)
    return interpreted, context


In [6]:
# Clear previous results so a failed rerun cannot leave stale policy context.
interpreted_query = policy_context = None
interpreted_query, policy_context = interpret_and_retrieve(
    user_query, client, policy_graph,
)
print("INTERPRETED QUERY")
print(interpreted_query.model_dump_json(indent=2))
print("\nRETRIEVED POLICY")
print(policy_context.model_dump_json(indent=2))


INTERPRETED QUERY
{
  "goal": "check cancellation eligibility",
  "operation": {
    "intent": "retrieve",
    "action": "check_cancellation_eligibility",
    "target": "reservation",
    "requested_properties": [
      {
        "entity": "reservation",
        "property": "reservation_id"
      }
    ]
  },
  "entities": [
    {
      "alias": "reservation",
      "type": "Reservation",
      "identifier": "ABC123"
    }
  ],
  "constraints": [],
  "clarifications": []
}

RETRIEVED POLICY
{
  "operation": "check_cancellation_eligibility",
  "action_node": "CancelReservation",
  "target_alias": "reservation",
  "conditions": [
    {
      "node_id": "DepartureBeyond24Hours",
      "properties": {
        "type": "Condition",
        "description": "Departure is more than 24 hours away",
        "attribute": "HoursUntilDeparture",
        "operator": ">",
        "value": 24
      },
      "checked_attributes": [
        "HoursUntilDeparture"
      ]
    },
    {
      "node_id": "Exis

The controller returns policy context only after query validation and graph coverage checks. Any future planner must be called after this stage succeeds. Coverage is checked against the current graph, not every rule in `policy.md`. Eligibility questions retrieve authorization rules for explanation, but authorization remains separate from eligibility. Runtime facts and confirmation must be checked before execution.

## Fake reservation database and lookup

Run these cells after interpretation and policy retrieval. Re-running the setup cell resets the in-memory records. The lookup returns facts; it does not evaluate eligibility or cancel a reservation.

In [7]:
import json
from experiments.airline.tools import Reservation, ReservationStore

store = ReservationStore([
    Reservation(
        reservation_id="ABC123", status="confirmed",
        refundable=True, hours_until_departure=48,
    ),
    Reservation(
        reservation_id="DEF456", status="confirmed",
        refundable=False, hours_until_departure=12,
    ),
])


def get_reservation(reservation_id: str) -> dict:
    """Retrieve authoritative reservation facts by reservation ID."""
    return store.get_reservation(reservation_id)


# Available for a future tool-calling stage; this experiment calls it directly.
reservation_lookup_tools = {"get_reservation": get_reservation}


In [8]:
def lookup_interpreted_reservation(query: InterpretedQuery, graph) -> dict:
    # Enforce policy retrieval even if this function is called independently.
    retrieve_policies(query, graph)
    target = next(
        entity for entity in query.entities
        if entity.alias == query.operation.target
    )
    if target.identifier is None or not target.identifier.strip():
        return {
            "success": False,
            "requires_clarification": True,
            "question": "What is your reservation ID?",
        }
    return get_reservation(target.identifier)


# Clear prior facts before attempting a fresh lookup.
reservation_data = None
if interpreted_query is None or policy_context is None:
    raise RuntimeError("Run interpretation and policy retrieval successfully first.")

reservation_data = lookup_interpreted_reservation(interpreted_query, policy_graph)
print(json.dumps(reservation_data, indent=2))


{
  "success": true,
  "reservation_id": "ABC123",
  "status": "confirmed",
  "refundable": true,
  "hours_until_departure": 48
}


## Deterministic evaluation

Rules come from the retrieved graph. Authorization is reported separately from eligibility; this first evaluation has no passenger confirmation.

In [9]:
from experiments.airline.notebook_workflow import (
    evaluate_policy, NotebookSession, run_customer_session,
)

if policy_context is None or reservation_data is None:
    raise RuntimeError("Run policy retrieval and reservation lookup first.")
policy_evaluation = evaluate_policy(policy_context, reservation_data, authorized=False)
print(json.dumps(policy_evaluation, indent=2))


{
  "conditions": [
    {
      "condition": "DepartureBeyond24Hours",
      "description": "Departure is more than 24 hours away",
      "category": "eligibility",
      "passed": true,
      "actual": 48,
      "error": null
    },
    {
      "condition": "Exists",
      "description": "Reservation exists",
      "category": "eligibility",
      "passed": true,
      "actual": true,
      "error": null
    },
    {
      "condition": "HasAuthorization",
      "description": "Passenger explicitly confirmed this cancellation",
      "category": "authorization",
      "passed": false,
      "actual": false,
      "error": null
    },
    {
      "condition": "IsConfirmed",
      "description": "Reservation is confirmed",
      "category": "eligibility",
      "passed": true,
      "actual": "confirmed",
      "error": null
    },
    {
      "condition": "IsRefundable",
      "description": "Reservation is refundable",
      "category": "eligibility",
      "passed": true,
      "actua

## Conversation experiment: manual or simulated customer

Both modes use the same LLM interpreter, deterministic retrieval, evaluator, and controller. Replies are controller-generated; there is no LLM planner yet. Each new session gets a fresh database. `ABC123` is eligible, `DEF456` is nonrefundable, and `GHI789` departs too soon.

Confirmation accepts only `yes` or `confirm ABC123` after a confirmation request. Other messages clear pending confirmation and are interpreted as a new turn. This deliberately narrow rule is not a general natural-language authorization evaluator.

In [10]:
def interpret_customer_message(message, history):
    response = client.chat(
        model=MODEL,
        messages=[{"role": "system", "content": QUERY_PROMPT}] + history + [
            {"role": "user", "content": message}
        ],
        format=InterpretedQuery.model_json_schema(),
        think=False,
    )
    return InterpretedQuery.model_validate_json(response.message.content)


def new_session():
    return NotebookSession(interpret_customer_message, retrieve_policies, policy_graph)


# Set False to skip the conversation when using Run All.
RUN_CONVERSATION = True
MODE = "simulated"  # Change to "simulated" for an LLM customer.
USER_MODEL = MODEL
CUSTOMER_GOAL = (
    "Cancel reservation ABC123. If asked to confirm cancellation of ABC123, "
    "reply yes. Wait for the agent to report the outcome before ending."
)
# Used only for the simulated example; other records must remain unchanged.
EXPECTED = {"ABC123": {"status": "cancelled"}}
MAX_TURNS = 8

conversation_report = None
if RUN_CONVERSATION:
    print(f"Starting {MODE} conversation. Waiting for Ollama may take a moment...", flush=True)
    session = new_session()
    conversation_report = run_customer_session(
        session, mode=MODE, client=client, user_model=USER_MODEL,
        goal=CUSTOMER_GOAL, max_turns=MAX_TURNS,
        expected=EXPECTED if MODE == "simulated" else None,
    )
    print(json.dumps(conversation_report, indent=2))
else:
    print("Conversation skipped. Set RUN_CONVERSATION = True to start.")


Starting simulated conversation. Waiting for Ollama may take a moment...
Waiting for simulated customer (qwen3:8b-q4_K_M)...
Customer: I would like to cancel my reservation with the ID ABC123.
Processing customer message...
Agent: Please confirm cancellation of ABC123. Reply 'yes' or 'confirm ABC123', or 'no' to decline.
Waiting for simulated customer (qwen3:8b-q4_K_M)...
Customer: yes
Processing customer message...
Agent: Reservation ABC123 was cancelled successfully.
{
  "initial": {
    "ABC123": {
      "reservation_id": "ABC123",
      "status": "confirmed",
      "refundable": true,
      "hours_until_departure": 48
    },
    "DEF456": {
      "reservation_id": "DEF456",
      "status": "confirmed",
      "refundable": false,
      "hours_until_departure": 48
    },
    "GHI789": {
      "reservation_id": "GHI789",
      "status": "confirmed",
      "refundable": true,
      "hours_until_departure": 12
    }
  },
  "final": {
    "ABC123": {
      "reservation_id": "ABC123",
   

For manual mode, set `RUN_CONVERSATION = True`, keep `MODE = "manual"`, and run the cell. Enter `/quit` to stop. For simulated mode, use `MODE = "simulated"`; adjust both `CUSTOMER_GOAL` and `EXPECTED` for each scenario. The report includes transcript, policy evidence, condition evaluations, state snapshots, and termination. `state_success` checks final state including unchanged unrelated records; it is not a full conversational-quality score.

Alternatively, run individual turns in separate cells without an input loop:
```python
manual_session = new_session()
print(manual_session.reply("Cancel reservation ABC123"))
# In a later cell:
print(manual_session.reply("yes"))
print(json.dumps(manual_session.report(), indent=2))
```

## Controlled evaluation: six scenarios × three runs

Each episode starts with fresh records. Checks cover state, policy retrieval and cancellation guards, scenario-specific response behavior, and termination. Response checks are specific to this controller, not an independent semantic judge. Simulator protocol corrections are counted separately. Raw transcripts and scores are saved per episode. A failed simulator or interpreter call counts as a failed run.

These are single-task episodes: the controller ends after cancellation, refusal, an explicit decline, or an eligibility answer. `task_completed` means the interaction ended; `outcome` records what happened. A pass requires the expected outcome, not just termination. Start a new session for another request.


In [11]:
import importlib
from datetime import datetime
from IPython.display import Markdown, display
import experiments.airline.notebook_workflow as workflow
import experiments.airline.notebook_scenarios as scenarios
importlib.reload(workflow)
importlib.reload(scenarios)

RUN_SUITE = True  # Set True to run 18 live episodes.
REPEATS = 3
suite_rows = None
if RUN_SUITE:
    suite_output = Path.cwd() / "results" / (
        "controlled-" + datetime.now().strftime("%Y%m%d-%H%M%S-%f")
    )
    suite_rows = scenarios.run_suite(
        interpret_customer_message, retrieve_policies, policy_graph,
        client, USER_MODEL, suite_output, repeats=REPEATS, max_turns=8,
    )
    table = ["| Scenario | Passed | State | Policy | Response | Termination | Corrections |",
             "|---|---|---|---|---|---|---|"]
    for spec in scenarios.SCENARIOS:
        rows = [r for r in suite_rows if r["scenario"] == spec["name"]]
        counts = [f"{sum(r[key] for r in rows)}/{len(rows)}" for key in
                  ("passed", "state_ok", "policy_ok", "response_ok", "termination_ok")]
        table.append("| " + " | ".join([spec["name"], *counts,
                     str(sum(r["protocol_corrections"] for r in rows))]) + " |")
    display(Markdown("\n".join(table)))
    print(f"Saved all episode details to {suite_output}")
else:
    print("Set RUN_SUITE = True to evaluate six scenarios, three runs each.")


Running eligible (1/3)...
  PASS (task_completed)
Running eligible (2/3)...
  PASS (task_completed)
Running eligible (3/3)...
  PASS (task_completed)
Running nonrefundable (1/3)...
  PASS (task_completed)
Running nonrefundable (2/3)...
  PASS (task_completed)
Running nonrefundable (3/3)...
  PASS (task_completed)
Running exactly_24_hours (1/3)...
  PASS (task_completed)
Running exactly_24_hours (2/3)...
  PASS (task_completed)
Running exactly_24_hours (3/3)...
  PASS (task_completed)
Running declined (1/3)...
  PASS (task_completed)
Running declined (2/3)...
  PASS (task_completed)
Running declined (3/3)...
  PASS (task_completed)
Running unknown (1/3)...
  PASS (task_completed)
Running unknown (2/3)...
  PASS (task_completed)
Running unknown (3/3)...
  PASS (task_completed)
Running eligibility_only (1/3)...
  PASS (task_completed)
Running eligibility_only (2/3)...
  PASS (task_completed)
Running eligibility_only (3/3)...
  PASS (task_completed)


| Scenario | Passed | State | Policy | Response | Termination | Corrections |
|---|---|---|---|---|---|---|
| eligible | 3/3 | 3/3 | 3/3 | 3/3 | 3/3 | 0 |
| nonrefundable | 3/3 | 3/3 | 3/3 | 3/3 | 3/3 | 0 |
| exactly_24_hours | 3/3 | 3/3 | 3/3 | 3/3 | 3/3 | 0 |
| declined | 3/3 | 3/3 | 3/3 | 3/3 | 3/3 | 0 |
| unknown | 3/3 | 3/3 | 3/3 | 3/3 | 3/3 | 0 |
| eligibility_only | 3/3 | 3/3 | 3/3 | 3/3 | 3/3 | 0 |

Saved all episode details to /Users/cahree/dev/Odysseus/experiments/airline/results/controlled-20261102-005111-267049


## Full-prose baseline

The agent receives the entire `policy.md` and lookup/cancellation tools. It does not use structured interpretation, graph retrieval, or the graph controller. Same six tasks, models, fresh data, and repeats. Store eligibility protections remain active; unauthorized or ineligible attempts are scored as violations even if blocked.

This compares complete systems, not only policy representations. Both conditions are compared with `observable-v1` scoring; the older graph-specific scores are preserved. Response checks are lexical proxies, and prose stopping uses a bounded confirmation-question heuristic. Review transcripts for borderline cases.

In [ ]:
import importlib
from datetime import datetime
from pathlib import Path
from IPython.display import Markdown, display
import experiments.airline.prose_baseline as prose
importlib.reload(prose)

RUN_PROSE = True  # Set True to run six scenarios x three repeats.
PROSE_REPEATS = 3
GRAPH_RESULTS = Path("/Users/cahree/dev/Odysseus/experiments/airline/results/controlled-20261102-005111-267049")
prose_rows = None
if RUN_PROSE:
    prose_output = Path.cwd() / "results" / ("prose-" + datetime.now().strftime("%Y%m%d-%H%M%S-%f"))
    prose_rows = prose.run_prose_suite(client, MODEL, USER_MODEL, prose_output,
                                     repeats=PROSE_REPEATS, max_turns=8)
    graph_comparison = prose.rescore_graph_run(GRAPH_RESULTS)
    table = ["| Scenario | Graph (shared scorer) | Prose |",
             "|---|---|---|"]
    for spec in prose.SCENARIOS:
        counts = []
        for rows in (graph_comparison, prose_rows):
            group = [r for r in rows if r["scenario"] == spec["name"]]
            counts.append(f"{sum(r['passed'] for r in group)}/{len(group)}")
        table.append("| " + " | ".join([spec["name"], *counts]) + " |")
    display(Markdown("\n".join(table)))
    comparison = {"scorer": "observable-v1", "graph_source": str(GRAPH_RESULTS),
                  "graph_metrics": prose.pass_metrics(graph_comparison),
                  "prose_metrics": prose.pass_metrics(prose_rows),
                  "graph_rescored_rows": graph_comparison}
    (prose_output / "comparison.json").write_text(json.dumps(comparison, indent=2))
    print(json.dumps({k:v for k,v in comparison.items() if k.endswith("metrics")}, indent=2))
    print(f"Saved baseline transcripts and comparison to {prose_output}")
else:
    print("Set RUN_PROSE = True to run the prose baseline.")


Prose: eligible (1/3)...
  FAIL (task_completed)
Prose: eligible (2/3)...
  FAIL (task_completed)
Prose: eligible (3/3)...
  FAIL (task_completed)
Prose: nonrefundable (1/3)...
  PASS (task_completed)
Prose: nonrefundable (2/3)...
  PASS (task_completed)
Prose: nonrefundable (3/3)...
  PASS (task_completed)
Prose: exactly_24_hours (1/3)...
  PASS (task_completed)
Prose: exactly_24_hours (2/3)...
  PASS (task_completed)
Prose: exactly_24_hours (3/3)...
  PASS (task_completed)
Prose: declined (1/3)...
  PASS (task_completed)
Prose: declined (2/3)...
  PASS (task_completed)
Prose: declined (3/3)...
  PASS (task_completed)
Prose: unknown (1/3)...
  PASS (task_completed)
Prose: unknown (2/3)...
  PASS (task_completed)
Prose: unknown (3/3)...
  PASS (task_completed)
Prose: eligibility_only (1/3)...


## Graph retrieval without deterministic enforcement
Mandatory retrieval and coverage checks, followed by the prose baseline LLM tool loop. No runtime policy evaluator or authorization gate. Store eligibility checks remain. This compares complete systems, not just validator presence.


In [14]:
# Paste this into a notebook cell after the interpretation/retrieval definitions.
import importlib
import json
from datetime import datetime
from pathlib import Path
from IPython.display import Markdown, display
import experiments.airline.prose_baseline as prose
import experiments.airline.graph_baseline as graph_baseline
importlib.reload(prose)
importlib.reload(graph_baseline)

RUN_GRAPH_NO_VALIDATOR = True  # Change to True to run.
GRAPH_NO_VALIDATOR_REPEATS = 3
CONTROLLED_RESULTS = Path('/Users/cahree/dev/Odysseus/experiments/airline/results/controlled-20261102-005111-267049')
PROSE_RESULTS = globals().get('prose_output', None)  # Or Path('/path/to/prose-run').

if RUN_GRAPH_NO_VALIDATOR:
    graph_no_validator_output = Path.cwd() / 'results' / (
        'graph-no-enforcement-' + datetime.now().strftime('%Y%m%d-%H%M%S-%f')
    )
    graph_no_validator_rows = graph_baseline.run_graph_suite(
        interpret_customer_message, retrieve_policies, policy_graph, client,
        MODEL, USER_MODEL, graph_no_validator_output,
        repeats=GRAPH_NO_VALIDATOR_REPEATS, max_turns=8,
    )
    conditions = {
        'Controlled graph': prose.rescore_graph_run(CONTROLLED_RESULTS),
        'Graph without enforcement': graph_no_validator_rows,
    }
    if PROSE_RESULTS is not None:
        conditions['Prose'] = prose.rescore_graph_run(Path(PROSE_RESULTS))
    else:
        print('Set PROSE_RESULTS to a saved prose folder to compare all three conditions.')
    table = ['| Scenario | ' + ' | '.join(conditions) + ' |',
             '|---|' + '---|' * len(conditions)]
    for spec in prose.SCENARIOS:
        counts = []
        for rows in conditions.values():
            group = [r for r in rows if r['scenario'] == spec['name']]
            counts.append(f"{sum(r['passed'] for r in group)}/{len(group)}")
        table.append('| ' + ' | '.join([spec['name'], *counts]) + ' |')
    display(Markdown('\n'.join(table)))
    metrics = {name: prose.pass_metrics(rows) for name, rows in conditions.items()}
    comparison = {'scorer': 'observable-v1', 'metrics': metrics,
                  'controlled_source': str(CONTROLLED_RESULTS),
                  'prose_source': str(PROSE_RESULTS) if PROSE_RESULTS else None,
                  'rescored_rows': conditions}
    (graph_no_validator_output / 'comparison.json').write_text(json.dumps(comparison, indent=2))
    print(json.dumps(metrics, indent=2))
    print(f'Saved reports to {graph_no_validator_output}')
else:
    print('Set RUN_GRAPH_NO_VALIDATOR = True to run this condition.')


Graph without enforcement: eligible (1/3)...
  FAIL (task_completed)
Graph without enforcement: eligible (2/3)...
  FAIL (task_completed)
Graph without enforcement: eligible (3/3)...
  FAIL (task_completed)
Graph without enforcement: nonrefundable (1/3)...
  FAIL (error)
Graph without enforcement: nonrefundable (2/3)...
  FAIL (error)
Graph without enforcement: nonrefundable (3/3)...
  FAIL (error)
Graph without enforcement: exactly_24_hours (1/3)...
  FAIL (task_completed)
Graph without enforcement: exactly_24_hours (2/3)...
  FAIL (task_completed)
Graph without enforcement: exactly_24_hours (3/3)...
  FAIL (task_completed)
Graph without enforcement: declined (1/3)...
  FAIL (task_completed)
Graph without enforcement: declined (2/3)...
  FAIL (error)
Graph without enforcement: declined (3/3)...
  FAIL (task_completed)
Graph without enforcement: unknown (1/3)...
  PASS (task_completed)
Graph without enforcement: unknown (2/3)...
  PASS (task_completed)
Graph without enforcement: unknow

| Scenario | Controlled graph | Graph without enforcement | Prose |
|---|---|---|---|
| eligible | 3/3 | 0/3 | 0/3 |
| nonrefundable | 3/3 | 0/3 | 3/3 |
| exactly_24_hours | 3/3 | 0/3 | 3/3 |
| declined | 3/3 | 0/3 | 3/3 |
| unknown | 3/3 | 3/3 | 3/3 |
| eligibility_only | 3/3 | 2/3 | 2/3 |

{
  "Controlled graph": {
    "pass^1": 1.0,
    "pass^2": 1.0,
    "pass^3": 1.0
  },
  "Graph without enforcement": {
    "pass^1": 0.27777777777777773,
    "pass^2": 0.2222222222222222,
    "pass^3": 0.16666666666666666
  },
  "Prose": {
    "pass^1": 0.7777777777777778,
    "pass^2": 0.7222222222222222,
    "pass^3": 0.6666666666666666
  }
}
Saved reports to /Users/cahree/dev/Odysseus/experiments/airline/results/graph-no-enforcement-20261102-011500-680647
